# Intelligent Crime Detective Platform — Person B Analytics Notebook
## Day 2: Dataset Cleaning & Feature Engineering

**Role:** Person B (Data Engineering, Geospatial Profiling, Feature Engineering & Classification Models)  
**Phase:** Day 2 of 10-Day Development Plan  

### Core Governance & Methodological Principles for Day 2
1. **Raw Data Immutability:** Raw datasets in `data/raw/` are immutable primary sources and are never altered.
2. **No Premature ML:** In accordance with the project roadmap, no model training, PCA, LWR, ID3, Naive Bayes, or k-NN algorithms are executed today.
3. **Target Leakage Prevention:** `disposition` and its derived indicator `is_solved` represent ground-truth investigative outcomes and are strictly isolated as target variables, never input features.
4. **Semantic Preservation:** Missing coordinates and unknown ages are never filled with `0`. Infant age `0` is strictly preserved as valid.
5. **Clean Processed Outputs:** Cleaned datasets with documented lineage are saved under `data/processed/`.

## 1. Environment Setup

Initialize runtime environment, configure paths, and import foundational libraries and internal utilities.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, OUTPUTS_DIR
from src.data_inspection import inspect_dataset
from src.data_cleaning import (
    load_csv, strip_whitespace, normalize_categorical,
    detect_duplicates, report_missing_values, parse_dates_safely,
    coerce_numeric, validate_coordinates, identify_total_rows
)
from src.data_quality import dataset_summary, duplicate_report, profile_columns, generate_quality_report, missing_value_report
from src.feature_engineering import (
    extract_temporal_features, create_target_solvability,
    flag_target_leakage_columns, standardize_tamil_nadu_districts
)
from src.run_cleaning_pipeline import run_all_cleaning

print(f"Project Root: {PROJECT_ROOT}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

## 2. Dataset Discovery

Programmatically discover and verify the physical files present in `data/raw/`.

In [ ]:
raw_files = sorted(list(RAW_DATA_DIR.glob("*.csv")))
print(f"Discovered {len(raw_files)} CSV datasets in {RAW_DATA_DIR}:")

discovery_records = []
for f in raw_files:
    size_kb = round(f.stat().st_size / 1024, 2)
    discovery_records.append({
        "Filename": f.name,
        "File Size (KB)": size_kb,
        "File Size (MB)": round(size_kb / 1024, 2),
    })

df_discovery = pd.DataFrame(discovery_records)
display(df_discovery)

## 3. Dataset Inventory

Define the analytical scope, target domain, and planned Person B modeling modules for each discovered dataset.

In [ ]:
inventory_data = [
    {
        "Filename": "homicide-data.csv",
        "Target Domain": "US Major City Homicides (50 cities)",
        "Target / Outcome Column": "disposition (Closed by arrest vs others)",
        "Planned Person B Modules": "Solvability Estimation (ID3, Naive Bayes, k-NN), Geographic Profiling (LWR)",
    },
    {
        "Filename": "dstrIPC_1_2014.csv",
        "Target Domain": "NCRB National District IPC Crime Records (2014)",
        "Target / Outcome Column": "Total Cognizable IPC crimes (Aggregate Benchmark)",
        "Planned Person B Modules": "Principal Component Analysis (PCA), Crime Offense Pattern Profiling",
    },
    {
        "Filename": "TN-2020-2022-total.csv",
        "Target Domain": "Tamil Nadu Multi-Year Cognizable Crime Totals (2020-2022)",
        "Target / Outcome Column": "Rate of Cognizable crime (IPC+SLL)",
        "Planned Person B Modules": "Longitudinal Trend Analysis, District Growth Comparison, Streamlit UI",
    },
    {
        "Filename": "TN-murder-2023.csv",
        "Target Domain": "Tamil Nadu Violent & Negligent Fatalities (2023)",
        "Target / Outcome Column": "Murder - Rate / Incidence",
        "Planned Person B Modules": "Tamil Nadu Geospatial Analytics, Folium Interactive Map",
    },
]

df_inventory = pd.DataFrame(inventory_data)
display(df_inventory)

## 4. Schema Inspection

Inspect the native dimensions, column headers, and data types across each source dataset.

In [ ]:
schema_summaries = []
loaded_raw = {}

for f in raw_files:
    enc = "latin-1" if "homicide" in f.name else "utf-8"
    df_temp = load_csv(f, encodings_to_try=[enc, "latin-1", "utf-8"])
    loaded_raw[f.name] = df_temp
    
    schema_summaries.append({
        "Dataset": f.name,
        "Rows": len(df_temp),
        "Columns": len(df_temp.columns),
        "Numeric Cols": len(df_temp.select_dtypes(include=[np.number]).columns),
        "String/Object Cols": len(df_temp.select_dtypes(include=["object", "string"]).columns),
    })

display(pd.DataFrame(schema_summaries))

## 5. Data Quality Analysis

Generate comprehensive quality audits across the raw data assets.

In [ ]:
quality_metrics = []
for name, df_temp in loaded_raw.items():
    summary = dataset_summary(df_temp, dataset_name=name)
    quality_metrics.append(summary)

display(pd.DataFrame(quality_metrics))

## 6. Missing Value Analysis

Audit missing value counts, percentages, and semantic causes across all fields.

In [ ]:
for name, df_temp in loaded_raw.items():
    rep = missing_value_report(df_temp)
    missing_cols = rep[rep["missing_count"] > 0]
    print(f"=== Missing Values in {name} ===")
    if len(missing_cols) == 0:
        print("  -> No missing values found in native DataFrame.")
    else:
        display(missing_cols)

print("\nSemantic Missingness Audit:")
print("1. homicide-data.csv: lat & lon have 60 nulls (0.11%). victim_age has 2,999 'Unknown' entries.")
print("2. TN-2020-2022-total.csv: Avadi and Tambaram have 'N/C' (Not Created) in 2020 & 2021.")
print("3. TN-murder-2023.csv: Specialized units (Railways, Cyber Cell) have '-' rates.")

## 7. Duplicate Analysis

Examine exact duplicate rows and test candidate primary keys for collisions.

In [ ]:
dup_results = []
for name, df_temp in loaded_raw.items():
    id_col = "uid" if "uid" in df_temp.columns else ("Sl No" if "Sl No" in df_temp.columns else None)
    d_rep = duplicate_report(df_temp, id_col=id_col)
    dup_results.append({
        "Dataset": name,
        "Exact Duplicate Rows": d_rep["exact_duplicate_rows"],
        "ID Column": d_rep["id_column"],
        "Duplicate IDs Count": d_rep["duplicate_ids_count"],
    })

display(pd.DataFrame(dup_results))

## 8. Categorical Feature Analysis

Profile categorical values, label cardinality, and distributions.

In [ ]:
df_hom = loaded_raw["homicide-data.csv"]
print("Victim Sex Distribution:")
display(df_hom["victim_sex"].value_counts(dropna=False).to_frame(name="Count"))

print("\nVictim Race Distribution:")
display(df_hom["victim_race"].value_counts(dropna=False).to_frame(name="Count"))

print("\nCase Disposition (Ground-Truth Clearance):")
display(df_hom["disposition"].value_counts(dropna=False).to_frame(name="Count"))

## 9. Numerical Feature Analysis

Inspect numerical distributions, boundary limits, and anomalies.

In [ ]:
# Inspect victim age distribution in homicide data
numeric_age = coerce_numeric(df_hom["victim_age"], sentinel_strings=["Unknown"])
print("Victim Age Statistics (excluding 'Unknown'):")
display(numeric_age.describe().to_frame(name="Victim Age Distribution"))

print(f"Infant records (Age == 0): {(numeric_age == 0).sum()} cases (preserved as valid)")
print(f"Unknown age records: {numeric_age.isnull().sum()} cases (represented as NaN)")

## 10. Geographic Feature Analysis

Validate latitude/longitude coordinate bounds and analyze district jurisdiction nomenclature.

In [ ]:
# Validate coordinates in homicide data
valid_coords = validate_coordinates(df_hom, lat_col="lat", lon_col="lon")
print("Coordinate Validity Audit:")
print(f"  Valid coordinates: {valid_coords.sum():,} ({valid_coords.mean()*100:.2f}%)")
print(f"  Missing/Invalid coordinates: {(~valid_coords).sum()} (0.11%)")
print(f"  Latitude range (valid): [{df_hom.loc[valid_coords, 'lat'].min():.4f}, {df_hom.loc[valid_coords, 'lat'].max():.4f}]")
print(f"  Longitude range (valid): [{df_hom.loc[valid_coords, 'lon'].min():.4f}, {df_hom.loc[valid_coords, 'lon'].max():.4f}]")

# Check Tamil Nadu district transliteration differences
df_tn_tot = loaded_raw["TN-2020-2022-total.csv"]
df_tn_mrd = loaded_raw["TN-murder-2023.csv"]
print("\nTamil Nadu District Jurisdictions in 2020-2022 Totals:", len(df_tn_tot))
print("Tamil Nadu District Jurisdictions in 2023 Murder:", len(df_tn_mrd))
new_dist = set(df_tn_mrd["Districts/City"]) - set(df_tn_tot["Districts"])
print("New district appearing in 2023:", new_dist)

## 11. Temporal Feature Analysis

Analyze reporting dates, handle entry typos safely, and extract calendar features.

In [ ]:
dates_raw = df_hom["reported_date"].astype(str)
print("Date string length distribution:")
display(dates_raw.str.len().value_counts().to_frame(name="Count"))

malformed = df_hom[dates_raw.str.len() != 8]
print(f"Malformed date records count: {len(malformed)}")
display(malformed[["uid", "reported_date", "city", "state"]])

dates_parsed = parse_dates_safely(df_hom["reported_date"], format="%Y%m%d", errors="coerce")
print(f"Successfully parsed dates: {dates_parsed.notnull().sum():,}")
print(f"NaT dates (safely coerced): {dates_parsed.isnull().sum()}")

## 12. Cleaning Decisions

Document the explicit, auditable rationale behind all transformations performed.

In [ ]:
decisions = [
    {
        "Aspect": "Missing Coordinates (homicide-data)",
        "Action": "Do not fill with 0. Flag via valid_coords and create homicide_spatial_clean.csv.",
        "Rationale": "Filling coordinates with 0 creates fictitious incidents at (0, 0) Off Africa, corrupting GIS maps and LWR."
    },
    {
        "Aspect": "Unknown Victim Age (homicide-data)",
        "Action": "Coerce 'Unknown' to NaN. Preserve age 0.",
        "Rationale": "0 is a valid numerical age for infants (<1 year old). Imputing Unknown with 0 introduces profound bias."
    },
    {
        "Aspect": "Malformed Dates (homicide-data)",
        "Action": "Safely coerce 9-digit integers to NaT while preserving raw reported_date.",
        "Rationale": "Prevents pipeline runtime crashes while maintaining data integrity."
    },
    {
        "Aspect": "Not Created 'N/C' (TN-2020-2022-total)",
        "Action": "Convert 'N/C' to NaN in Avadi and Tambaram for 2020/2021.",
        "Rationale": "Districts created in late 2021 lack separate historical figures. Conversion allows numeric calculations."
    },
    {
        "Aspect": "Special Police Units Rates '-' (TN-murder-2023)",
        "Action": "Convert '-' to NaN in rate columns.",
        "Rationale": "Railways and Cyber Cell lack residential population bases, making per-lakh rates undefined."
    },
    {
        "Aspect": "Aggregate Summary Rows (NCRB & TN)",
        "Action": "Add is_total_row boolean flag.",
        "Rationale": "Prevents double-counting state crime totals during district-level statistical modeling."
    },
    {
        "Aspect": "Target Variable & Leakage",
        "Action": "Derive is_solved (1=Closed by arrest) and label strictly as TARGET ONLY.",
        "Rationale": "Case disposition is a post-investigation outcome; must never be supplied to input features (X)."
    },
]

display(pd.DataFrame(decisions))

## 13. Clean Dataset Generation

Execute the end-to-end data cleaning pipeline and serialize cleaned datasets into `data/processed/`.

In [ ]:
# Execute the modular pipeline runner
processed_outputs = run_all_cleaning()

processed_summary = []
for key, path in processed_outputs.items():
    df_p = pd.read_csv(path)
    processed_summary.append({
        "Dataset Identifier": key,
        "Output File": path.name,
        "Rows": len(df_p),
        "Columns": len(df_p.columns),
        "Size (KB)": round(path.stat().st_size / 1024, 2),
    })

display(pd.DataFrame(processed_summary))

## 14. Feature Dictionary

Preview the structured feature dictionary documented under `outputs/person_b_feature_dictionary.md`.

In [ ]:
feat_dict_path = OUTPUTS_DIR / "person_b_feature_dictionary.md"
if feat_dict_path.exists():
    print(f"Feature Dictionary successfully established at: {feat_dict_path}")
    with open(feat_dict_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print(f"Total lines of documentation: {len(lines)}")
    # Print first 35 lines preview
    print("".join(lines[:35]))
else:
    print("Feature dictionary not found!")

## 15. Final Data Quality Report

Verify the data health and integrity of all processed outputs.

In [ ]:
print("=== FINAL PROCESSED DATA QUALITY VERIFICATION ===")
for key, path in processed_outputs.items():
    df_p = pd.read_csv(path)
    print(f"\n--- {key} ({path.name}) ---")
    print(f"Rows: {len(df_p):,}, Columns: {len(df_p.columns)}")
    print(f"Duplicate Rows: {df_p.duplicated().sum()}")
    null_counts = df_p.isnull().sum()
    active_nulls = null_counts[null_counts > 0]
    if len(active_nulls) > 0:
        print("Null counts per column:")
        for col, cnt in active_nulls.items():
            print(f"  {col}: {cnt} ({cnt/len(df_p)*100:.2f}%)")
    else:
        print("  Zero null cells detected.")

print("\nDay 2 Dataset Cleaning & Feature Engineering successfully completed!")